<a href="https://colab.research.google.com/github/Josephleme/plp-da-a2-rainfall-advisory/blob/main/rainfall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import calendar
import csv

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [

    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],# Nairobi

    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],# Nakuru

    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],# Kisumu

    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],# Mombasa

]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ","9", "12", "", "45", "182", "96"]
months = list(calendar.month_name[1:])

def clean_machakos(raw_data):
    cleaned = []
    imputed_months = []

    for i, val in enumerate(raw_data):
        try:
            cleaned.append(int(val.strip()))
        except ValueError:
            cleaned.append(None)
            imputed_months.append(months[i])

    valid_only = [x for x in cleaned if x is not None]
    avg_val = round(sum(valid_only) / len(valid_only))
    final_clean = [avg_val if x is None else x for x in cleaned]

    print(f"Imputed months (replaced with {avg_val} mm): {imputed_months}")
    return final_clean

machakos_clean = clean_machakos(machakos_raw)
R = np.array(clean_rows + [machakos_clean])

print("Shape:", R.shape)
print("Dtype:", R.dtype)

Imputed months (replaced with 68 mm): ['April', 'September']
Shape: (5, 12)
Dtype: int64


In [12]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0).round(1)

print("Annual Totals (mm):")
for county, total in zip(counties, annual_totals):
    print(f"  {county}: {total}")

print("\nMonthly Averages (mm):")
for month, avg in zip(months, monthly_averages):
    print(f"  {month}: {avg}")

wettest_indices = R.argmax(axis=1)
wettest_months = [months[idx] for idx in wettest_indices]

print("\nWettest Month per County:")
for county, month in zip(counties, wettest_months):
    print(f"  {county}: {month}")

dry_mask = R < 50
dry_counts = dry_mask.sum(axis=1)

print("\nDry Months (< 50 mm) per County:")
for county, count in zip(counties, dry_counts):
    print(f"  {county}: {count}")

pairs = R[:, :-1] + R[:, 1:]
best_pair_indices = pairs.argmax(axis=1)
planting_windows = [f"{months[idx]}-{months[idx + 1]}" for idx in best_pair_indices]

print("\nRecommended Planting Windows:")
for county, window in zip(counties, planting_windows):
    print(f"  {county}: {window}")


Annual Totals (mm):
  Nairobi: 1045
  Nakuru: 916
  Kisumu: 1371
  Mombasa: 1075
  Machakos: 811

Monthly Averages (mm):
  January: 47.0
  February: 47.6
  March: 91.0
  April: 164.4
  May: 171.2
  June: 67.6
  July: 56.6
  August: 58.2
  September: 64.2
  October: 67.8
  November: 127.4
  December: 80.6

Wettest Month per County:
  Nairobi: April
  Nakuru: April
  Kisumu: April
  Mombasa: May
  Machakos: November

Dry Months (< 50 mm) per County:
  Nairobi: 4
  Nakuru: 3
  Kisumu: 0
  Mombasa: 2
  Machakos: 6

Recommended Planting Windows:
  Nairobi: April-May
  Nakuru: April-May
  Kisumu: April-May
  Mombasa: April-May
  Machakos: November-December


In [13]:
summary_rows = []
for county, total, wettest, dry, window in zip(
    counties, annual_totals, wettest_months, dry_counts, planting_windows
):
    summary_rows.append({
        "county": county,
        "annual_total_mm": int(total),
        "wettest_month": wettest,
        "dry_months": int(dry),
        "planting_window": window
    })

fieldnames = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

print("Saved rainfall_summary.csv successfully.")


Saved rainfall_summary.csv successfully.


## Planting Advisory

Farmers in Nairobi, Nakuru, Kisumu, and Mombasa should prepare for planting at the start of the March–April or April–May long rains, which represent the peak two-month rainfall windows for these regions. In contrast, Machakos experiences its strongest consecutive rainfall later in the year during November–December and faces seven dry months below 50 mm, requiring drought-tolerant crops and early water harvesting. Kisumu receives the highest annual rainfall (1,371 mm) with zero dry months, supporting longer-maturing crop varieties throughout the year. All SMS advisories should encourage farmers to plant just ahead of their county's peak two-month window to maximize germination and yield.

## Data Quality Notes

The raw rainfall dataset for Machakos contained two invalid text entries ("N/A" for April and an empty string for September) resulting from manual transcription. Both missing readings were imputed using the rounded mean of the ten valid Machakos months (68 mm), which may slightly understate April's typical long-rains peak and overstate September's dry-season baseline.